In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [8]:
web_search_schema = {"type": "web_search_20250305", "name": "web_search", "max_uses": 5, "allowed_domains":["nih.gov"]}

In [9]:
messages = []
add_user_message(
    messages,
    """
    Whats the best exercise for gaining leg muscles
    """,
)
response = chat(messages,tools=[web_search_schema])
response

Message(id='msg_01BF4a15XhQ42NpeAmWMpwGK', container=None, content=[TextBlock(citations=None, text='The best exercises for gaining leg muscle are compound movements that target multiple muscle groups. Here are the most effective ones:\n\n**1. Barbell Back Squat**\nOften called the "king" of leg exercises, squats work your quadriceps, hamstrings, glutes, and calves while also engaging your core. They allow you to lift heavy weights, which is crucial for muscle growth.\n\n**2. Deadlifts (Conventional or Romanian)**\nThese primarily target your hamstrings, glutes, and lower back. Romanian deadlifts are particularly effective for hamstring development.\n\n**3. Leg Press**\nA great alternative to squats that allows you to move heavy weight safely while focusing on quads, hamstrings, and glutes with less stress on your lower back.\n\n**4. Bulgarian Split Squats**\nAn excellent unilateral exercise that builds strength and muscle in each leg individually while improving balance. Targets quads,

In [10]:
# Get just Claude's text summary
for block in response.content:
    if block.type == "text":
        print(block.text)

The best exercises for gaining leg muscle are compound movements that target multiple muscle groups. Here are the most effective ones:

**1. Barbell Back Squat**
Often called the "king" of leg exercises, squats work your quadriceps, hamstrings, glutes, and calves while also engaging your core. They allow you to lift heavy weights, which is crucial for muscle growth.

**2. Deadlifts (Conventional or Romanian)**
These primarily target your hamstrings, glutes, and lower back. Romanian deadlifts are particularly effective for hamstring development.

**3. Leg Press**
A great alternative to squats that allows you to move heavy weight safely while focusing on quads, hamstrings, and glutes with less stress on your lower back.

**4. Bulgarian Split Squats**
An excellent unilateral exercise that builds strength and muscle in each leg individually while improving balance. Targets quads, glutes, and hamstrings.

**5. Lunges (Walking or Stationary)**
Another effective unilateral movement that works

In [ ]:
def extract_response(response):
    for block in response.content:
        if block.type == "text":
            print("📝 Claude says:\n", block.text)
        elif block.type == "tool_use":
            print("🔍 Searched for:", block.input)
        elif block.type == "web_search_result":
            print("🌐 Source:", block.title, "-", block.url)

extract_response(response)

In [7]:
import json

response_json = json.loads(response.model_dump_json())
print(json.dumps(response_json, indent=2))

{
  "id": "msg_01Qyiu9xBjemCCmRH53SqSVm",
  "container": null,
  "content": [
    {
      "id": "srvtoolu_01XGmHFcqeMjTBEQ9banXnb3",
      "caller": {
        "type": "direct"
      },
      "input": {
        "query": "AI news today March 2026"
      },
      "name": "web_search",
      "type": "server_tool_use"
    },
    {
      "caller": {
        "type": "direct"
      },
      "content": [
        {
          "encrypted_content": "Er8JCioIDRgCIiQyMDcxODJkZi0xMjdlLTQyYTItYTY0MC1lNWI4MGY4YTVlYWQSDOU3IaGwQHaBnMvm+BoMot5Xuh7y7g1nyokRIjAgO+d4x1/zrPd/5+gd57pmgNamuK6LbTiZI6NDABDfbjQlphCJVKbkurUJWLAgKWEqwgi1TSbXwhoaHkIOguYjtKIm3E+HldxNL50XgIY1Dz2YwpKxSocTG7U6GggzaU8+EWCP1P3wUKjGyk02oET+IgAc1KHQFnjT9UunYZTJIzJW9psbg5I6pa8zf6iRCBy9yv//ierYb0q4vrlwO+GbXFGy/LJXJUFSLjfcrcy/VzbzfYO33KXhfyWxbNsOwcDMcF0ZoxSP63sLc3XPYc1ZdnYDecTVzK3WO1lb0Et8+vmUmrvWbDa2rbUwVuBtHznsgDb9giCxlrfwFnBUdwW51ozVuMTUA9aXUw4BE9jzOcFCcZUomGtkKll7YMdvBKCRJT+yyiOu3btvfW63/1ldsW8s5QtJoz+Yl/l7obQLMdmv54bdDdn9XVhmo9ZW24xYBewCd/7